In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Task 1: Write your code here:
# Load datase

csv_path = os.path.join(path, "Q3_data.csv")

data = pd.read_csv(csv_path)

data.head()

In [ ]:
# Task 1: Write your code here:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# 1. Handle missing values (if any)
print("Missing values per column:")
print(data.isnull().sum())

# Simple strategy: fill numerical with median, categorical with mode
for col in data.columns:
    if data[col].dtype == 'object':
        data[col].fillna(data[col].mode()[0], inplace=True)
    else:
        data[col].fillna(data[col].median(), inplace=True)

In [ ]:
# Task 2: Write your code here:
# 2. Remove duplicates
duplicates = data.duplicated().sum()
print("Number of duplicate rows:", duplicates)
data = data.drop_duplicates()


In [ ]:
# Task 3: Write your code here:
# 3. Encode categorical variables if needed
# Check for object columns
categorical_cols = data.select_dtypes(include='object').columns
print("Categorical columns:", categorical_cols)

# Apply LabelEncoder if there are any
le = LabelEncoder()
for col in categorical_cols:
    data[col] = le.fit_transform(data[col])

In [ ]:
# Task 4: Write your code here:
# 4. Feature scaling (numerical)
numerical_cols = data.select_dtypes(include=['int64','float64']).drop('target', axis=1)
scaler = StandardScaler()
data[numerical_cols.columns] = scaler.fit_transform(data[numerical_cols.columns])



In [ ]:
# Task 5: Write your code here:
# 5. Check target imbalance
target_counts = data['target'].value_counts()
print("Target distribution:")
print(target_counts)

imbalance_ratio = target_counts.min() / target_counts.max()
if imbalance_ratio < 0.4:
    print("Target is imbalanced.")
else:
    print("Target is reasonably balanced.")

In [ ]:
# Task 1: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

# Features and target
X = data.drop('target', axis=1)
y = data['target']

# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostClassifier(verbose=0, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

print("Average Accuracy across folds:", np.mean(accuracies))


In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt

# Plot feature importance
feature_importances = model.get_feature_importance()
feature_names = X.columns

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(12,6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.gca().invert_yaxis()
plt.title("Feature Importances")
plt.show()

# Golden feature
golden_feature = importance_df.iloc[0]['feature']
print("Golden Feature:", golden_feature)


In [ ]:
# Task Bonus: Write your code here:

In [ ]:
X_golden = X[[golden_feature]]

accuracies_golden = []

for train_index, test_index in skf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_golden = CatBoostClassifier(verbose=0, random_state=42)
    model_golden.fit(X_train, y_train)

    y_pred = model_golden.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies_golden.append(acc)

print("Average Accuracy with Golden Feature only:", np.mean(accuracies_golden))
print("Performance comparison: Full model vs Golden feature only")
